# Modele zachowań, kalibracja i walidacja

Wyniki łańcucha `python -m src.model.pipeline` (zapisane w `processed/model/results.pkl`). Najpierw uruchom `python -m src.data.build`, potem pipeline (ok. 10 min), potem ten notebook.

Łańcuch:
1. **modele zachowań** dopasowane na dniach kalibracyjnych: popyt (ilu chętnych), cel podróży (dokąd), czas jazdy (jak długo),
2. **kalibracja** parametrów, których nie widać w danych (co robi klient przy pustej lub przepełnionej stacji),
3. **walidacja**: symulacja na późniejszych dniach, których modele nie widziały, porównana z rzeczywistością i z modelami bazowymi,
4. **hipotezy** H1, H4, H5a na modelu popytu ze wszystkich dni.

Kod: `src/model/` (opis w README).

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd() if (Path.cwd() / "processed").exists() else Path.cwd().parent
R = pickle.load(open(ROOT / "processed" / "model" / "results.pkl", "rb"))
stations = pd.read_parquet(ROOT / "processed" / "stations.parquet").set_index("station_id")
TZ = "Europe/Warsaw"

# Paleta jak w pozostałych notebookach: symulacja = niebieski, rzeczywistość = pomarańczowy
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "legend.frameon": False, "lines.linewidth": 2,
})

c, v = R["calib_days"], R["valid_days"]
print(f"Kalibracja: {c[0]} – {c[-1]} ({len(c)} dni), walidacja: {v[0]} – {v[-1]} ({len(v)} dni)")
if R["quick"]:
    print("UWAGA: pipeline uruchomiony z --quick (mało prób i replikacji), wyniki tylko do sprawdzenia łańcucha")
if R["weather_imputed"]:
    w = R["weather_imputed"]
    print(f"Godziny bez pogody z Open-Meteo (uzupełnione): {len(w)}, {min(w):%d.%m %H:%M} – {max(w):%d.%m %H:%M} UTC")

Kalibracja: 2026-10-23 – 2026-10-29 (7 dni), walidacja: 2026-10-30 – 2026-11-04 (6 dni)


## 1. Model popytu: ilu chętnych przychodzi na stację

**λ** to średnia liczba klientów, którzy w danej godzinie chcą wypożyczyć rower na danej stacji. Model to regresja ujemna dwumianowa (dla zliczeń) dopasowana do liczby wypożyczeń na stację × godzinę:

$$\log \lambda = \alpha_{\text{stacja},\,\text{pora dnia}} + \text{godzina} \times \text{typ dnia} + \beta\cdot\text{opad} + \beta\cdot T + \beta\cdot T^2 + \beta\cdot\text{zmrok} + \beta\cdot\text{zajęcia}\times\text{uczelnia} + \beta\cdot\text{korki}$$

- każda stacja ma własny poziom w każdej porze dnia (noc, rano, południe, popołudnie, wieczór), więc ma też własny kształt doby: jedne stacje pustoszeją rano, inne po południu; pory dnia z bardzo małą liczbą wypożyczeń (zwykle noc) dzielą wspólny poziom,
- godziny, w których stacja była pusta, nie liczą się jako „brak chętnych”: model liczy popyt na godzinę, w której rower był dostępny (offset), a godziny z < 15 min dostępności są pomijane,
- „dzień zajęć” sam w sobie nie wchodzi do modelu: w zebranych danych pokrywa się z dniem roboczym, więc nie da się ich rozdzielić.

Sprawdzenie na dniach walidacyjnych (godziny, w których stacja miała rowery):

In [2]:
d = R["demand"]
print(f"Stacje: {d['n_stations']}, nadmierna dyspersja alpha = {d['alpha']:.3f}")
ch = d["check"]
print(f"Wypożyczenia: rzeczywiste {ch['suma_rzeczywista']:,.0f}, prognoza {ch['suma_prognoza']:,.0f} "
      f"({ch['suma_prognoza'] / ch['suma_rzeczywista'] - 1:+.1%}); "
      f"korelacja godzina po godzinie w całym systemie: {ch['korelacja_system_h']:.2f}")

Stacje: 343, nadmierna dyspersja alpha = 0.245
Wypożyczenia: rzeczywiste 45,281, prognoza 49,434 (+9.2%); korelacja godzina po godzinie w całym systemie: 0.92


## 2. Cel podróży: macierz OD i model grawitacyjny

Model grawitacyjny: $P(j \mid i) \propto A_j \cdot e^{-\beta d_{ij}}$, gdzie $A_j$ to popularność stacji docelowej (udział przyjazdów w danej porze dnia), a $d_{ij}$ to odległość po drogach rowerowych. Powroty na tę samą stację mają osobną wagę. Symulacja używa macierzy z danych wygładzonej grawitacją: częste relacje jak w danych, rzadkie z modelu.

Porównanie na przejazdach z dni walidacyjnych (log-wiarygodność: wyżej = lepiej; CPC: 1 = identyczne przepływy):

In [3]:
o = R["od"]
print(f"beta = {o['beta']:.3f} / km (każdy km zmniejsza szansę wyboru celu {np.exp(o['beta']):.1f} raza); "
      f"dopasowanie na {o['n_trips']:,} przejazdach")
o["eval"].style.format("{:.3f}")

beta = 0.807 / km (każdy km zmniejsza szansę wyboru celu 2.2 raza); dopasowanie na 67,397 przejazdach


,loglik_na_przejazd,CPC
model,,
empiryczna,-4.736,0.398
grawitacyjna,-4.103,0.314
empiryczna + grawitacja (symulacja),-4.097,0.415


## 3. Czas jazdy

$\log(\text{czas}) \sim N(a + b \log(\text{km}) + c \cdot \text{e-rower}, \sigma)$. Czas w danych to różnica pomiarów co 5 min, więc traktujemy go jako przedział ±5 min. Powroty na tę samą stację mają osobny rozkład.

In [4]:
p = R["duration"]["params"]
print(f"a = {p['a']:.2f}, b = {p['b']:.2f} (2x dłuższa trasa = {2 ** p['b']:.2f}x dłuższy czas), "
      f"e-rower: {np.exp(p['c_ebike']) - 1:+.0%} czasu, sigma = {p['sigma']:.2f}")
R["duration"]["check"].to_frame("wartość").style.format("{:.2f}")

a = 2.46, b = 0.59 (2x dłuższa trasa = 1.51x dłuższy czas), e-rower: -15% czasu, sigma = 0.48


,wartość
udzial_do_20_min_dane,0.75
udzial_do_20_min_model,0.68
mediana_dane_min,15.00
mediana_model_min,15.00
predkosc_mediana_kmh,6.74


Rozkład log-normalny nie odtwarza w pełni spiętrzenia przed darmową 20. minutą (hipoteza A).

## 4. Kalibracja

Parametry klienta, których nie widać w danych, dobrane przez Optunę na dniach kalibracyjnych:
- `p_walk`: prawdopodobieństwo, że klient przy pustej stacji pójdzie do sąsiedniej (inaczej rezygnuje = utracony popyt),
- `radius_m`: jak daleko pójdzie (i jak daleko odjedzie od przepełnionej stacji),
- `overflow_k`: jak silnie przepełnienie zniechęca do zostawienia roweru.

Cel: RMSE zapełnienia stacji / odch. std + 0,5 · RMSE przejazdów na godzinę / średnia. Każda próba to kilka replikacji z tymi samymi ziarnami losowymi.

In [ ]:
cal = R["calibration"]
print({k: round(v, 3) for k, v in cal["params"].items()})
tr = cal["trials"]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.scatter(tr["number"], tr["value"], s=14, color=MUTED, label="próba")
ax.plot(tr["number"], tr["value"].cummin(), color=BLUE, label="najlepsza dotąd")
ax.set(xlabel="numer próby", ylabel="funkcja celu (niżej = lepiej)")
ax.set_title("Przebieg kalibracji", pad=24)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
plt.show()

## 5. Walidacja

Symulacja ze skalibrowanymi parametrami na dniach walidacyjnych. Każda doba startuje od rzeczywistego stanu stacji o północy, przewozy operatora są odtwarzane z danych.

In [ ]:
val = R["validation"]
s = val["sim"]
ids = val["station_ids"]
m = val["metrics"]
print(f"Przejazdy na godzinę: symulacja {m['trips_per_h_sim']:.0f}, rzeczywistość {m['trips_per_h_obs']:.0f}")
print(f"Czas pustej stacji: symulacja {m['empty_share_sim']:.1%}, rzeczywistość {m['empty_share_obs']:.1%} "
      f"(korelacja między stacjami {m['empty_share_corr_station']:.2f})")
print(f"Zgodność macierzy OD (CPC): {m['od_cpc']:.2f}; utracony popyt w symulacji: {m['lost_per_day']:.0f} klientów / dobę")

tab = val["baselines"].copy()
tab.loc["symulacja"] = [m["rmse_occ"], m["mae_occ"], m["rmse_trips_per_h"]]
tab.columns = ["RMSE zapełnienia [rowery]", "MAE zapełnienia [rowery]", "RMSE przejazdów / h"]
tab.style.format("{:.2f}", na_rep="–").highlight_min(axis=0, props="font-weight: bold")

Modele bazowe w tych samych chwilach: „dzień wcześniej” (ten sam moment poprzedniej doby), „średnia historyczna” (średnia z dni kalibracyjnych dla stacji, godziny i typu dnia) i „stan z północy” (ma tę samą informację na start co symulacja, więc to najuczciwsze porównanie).

Dzień po dniu:

In [ ]:
local_day = pd.Index(s["times"].tz_convert(TZ).date)
hour_day = pd.Index(s["hours"].tz_convert(TZ).date)
occ_mean = s["occ_sim"].mean(axis=0)
rows = {}
for day in pd.unique(local_day):
    k, h = local_day == day, hour_day == day
    sys_sim, sys_obs = s["dep_sim"][:, h].sum(axis=2).mean(axis=0), s["dep_obs"][h].sum(axis=1)
    rows[day] = {"RMSE zapełnienia": np.sqrt(np.mean((occ_mean[k] - s["occ_obs"][k]) ** 2)),
                 "przejazdy symulacja": sys_sim.sum(), "przejazdy rzeczywistość": sys_obs.sum(),
                 "pusta stacja symulacja": (s["occ_sim"][:, k] == 0).mean(),
                 "pusta stacja rzeczywistość": (s["occ_obs"][k] == 0).mean()}
pd.DataFrame(rows).T.style.format({"RMSE zapełnienia": "{:.2f}", "przejazdy symulacja": "{:,.0f}",
                                   "przejazdy rzeczywistość": "{:,.0f}", "pusta stacja symulacja": "{:.1%}",
                                   "pusta stacja rzeczywistość": "{:.1%}"})

### Zapełnienie wybranych stacji: model vs rzeczywistość

Pasmo to 95% replikacji symulacji. Skoki o północy to celowy powrót do rzeczywistego stanu.

In [ ]:
SHOW = ["2585762", "2585430", "2585445", "2585466", "52488396", "2585776"]
SHOW = [sid for sid in SHOW if sid in ids]
t = s["times"].tz_convert(TZ).tz_localize(None)
fig, axes = plt.subplots(len(SHOW), 1, figsize=(11, 2.3 * len(SHOW)), sharex=True)
for ax, sid in zip(axes, SHOW):
    j = ids.index(sid)
    lo, hi = np.percentile(s["occ_sim"][:, :, j], [2.5, 97.5], axis=0)
    ax.fill_between(t, lo, hi, color=BLUE, alpha=0.18, lw=0, label="symulacja: 95% replikacji")
    ax.plot(t, s["occ_sim"][:, :, j].mean(axis=0), color=BLUE, lw=1.2, label="symulacja: średnia")
    ax.plot(t, s["occ_obs"][:, j], color=ORANGE, lw=1.2, label="rzeczywistość")
    ax.set_title(stations.loc[sid, "name"], fontsize=10, loc="left")
    ax.set_ylabel("rowery")
    ax.set_ylim(0, None)
axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.15), ncol=3)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
fig.tight_layout()
plt.show()

In [ ]:
h = s["hours"].tz_convert(TZ).tz_localize(None)
sys_sim = s["dep_sim"].sum(axis=2)
lo, hi = np.percentile(sys_sim, [2.5, 97.5], axis=0)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.fill_between(h, lo, hi, color=BLUE, alpha=0.18, lw=0, label="symulacja: 95% replikacji")
ax.plot(h, sys_sim.mean(axis=0), color=BLUE, lw=1.5, label="symulacja: średnia")
ax.plot(h, s["dep_obs"].sum(axis=1), color=ORANGE, lw=1.5, label="rzeczywistość")
ax.set(ylabel="wypożyczenia / h", ylim=(0, None))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.set_title("Przejazdy na godzinę w całym systemie", pad=24)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3)
plt.show()

In [ ]:
es = (s["occ_sim"] == 0).mean(axis=(0, 1))
eo = (s["occ_obs"] == 0).mean(axis=0)
mx = max(es.max(), eo.max()) * 1.05
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, mx], [0, mx], color=MUTED, lw=1, ls="--")
ax.scatter(eo, es, s=14, alpha=0.6, color=BLUE)
ax.annotate("model = rzeczywistość", (mx * 0.62, mx * 0.55), color=INK2, fontsize=9, rotation=45)
ax.set(xlabel="rzeczywistość", ylabel="symulacja", xlim=(0, mx), ylim=(0, mx))
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.set_title("Odsetek czasu pustej stacji\n(każda kropka = stacja)", fontsize=11)
plt.show()

## 6. Hipotezy H1, H4, H5a (model popytu na wszystkich dniach)

IRR to mnożnik popytu na jednostkę zmiennej (1 = brak wpływu). Przedziały 95% i p pochodzą z błędów **klastrowanych po godzinie**: pogoda i korki są wspólne dla wszystkich stacji w danej godzinie, więc zwykłe błędy standardowe (traktujące każdą stację × godzinę jako niezależną) byłyby zbyt pewne siebie.

Jednostki: `rain` = log(1 + mm/h), `temp` = (T − 10 °C) / 5, `temp2` = jej kwadrat, `dark` = po zmroku, `classes_near_uni` = dzień zajęć × stacja z uczelnią w promieniu 300 m, `congestion` = wskaźnik korków (0–1), `saturday` = sobota względem niedzieli/święta.

In [ ]:
H = R["hypotheses"]
r = H["range"]
print(f"{H['n_obs']:,} obserwacji stacja × godzina, {H['n_days']} dni. Zakres: {r['temp_min']:.1f}–{r['temp_max']:.1f} °C, "
      f"{r['rain_hours']} z {r['hours']} godzin z opadem ≥ 0,1 mm, korki {r['congestion_min']:.2f}–{r['congestion_max']:.2f}")
H["coef"][["IRR", "CI_lo", "CI_hi", "p"]].style.format({"IRR": "{:.3f}", "CI_lo": "{:.3f}", "CI_hi": "{:.3f}", "p": "{:.3g}"})

In [ ]:
lr = H["lr_temp2"]
kind = "maksimum (odwrócone U)" if H["coef"].loc["temp2", "b"] < 0 else "minimum, nie maksimum"
print(f"H1: test LR członu T²: LR = {lr['LR']:.1f}, p = {lr['p']:.3g} (zakłada niezależność, więc p jest zaniżone)")
print(f"    wierzchołek paraboli przy {H['temp_vertex_C']:.1f} °C: to {kind}")

- **H1 (pogoda):** wpływ deszczu odczytujemy z `rain`. Temperatura zmienia się głównie między dniami, więc przy kilkunastu dniach jej efekt miesza się z innymi różnicami między dniami; „odwrócone U” wymaga danych z cieplejszych i mroźnych dni.
- **H5a (korki):** `congestion` przy kontroli godziny × typ dnia, więc to nie jest efekt samego szczytu.
- **H4 (kalendarz):** `classes_near_uni` mierzy na razie „stacja przy uczelni w dzień roboczy”, bo wszystkie zebrane dni robocze to dni zajęć. Rozstrzygnie dopiero przerwa świąteczna lub sesja.